In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchtsmixer import TSMixer
import os
import sys
import PIL
from PIL import Image
import numpy as np
from torchvision import transforms
from torchinfo import summary
import cv2

sys.path.append("../unisal/")
import unisal

sys.path.append("..")
import utils
from models import Conv_tsmixer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:

def preprocess(img):
    img = np.ascontiguousarray(img[:, :, ::-1])

    preproc_cfg = {
        "out_size": [288, 384],
        "rgb_mean": (0.485, 0.456, 0.406),
        "rgb_std": (0.229, 0.224, 0.225),
    }
    transformations = [
        transforms.ToPILImage(),
        transforms.Resize(preproc_cfg["out_size"], interpolation=PIL.Image.LANCZOS),
        transforms.ToTensor(),
    ]
    if "rgb_mean" in preproc_cfg:
        transformations.append(
            
            transforms.Normalize(preproc_cfg["rgb_mean"], preproc_cfg["rgb_std"])
        )
    processing = transforms.Compose(transformations)
    tensor = processing(img)
    return tensor

def sal_process(smap):
    smap = (smap - np.min(smap)) / ((np.max(smap) - np.min(smap)))
    smap = (smap - 0.5) / 0.5 # Scale to [-1,1]
    return smap    

In [3]:
import torch
import torch.nn as nn

class InputWrapper(nn.Module):
    def forward(self, x):
        if x.shape != (1, 3, 288, 384):
            raise ValueError(f"Expected input shape (1, 3, 288, 384), but got {x.shape}")
        return x.unsqueeze(0)

class WrappedUNISAL(nn.Module):
    def __init__(self, sal_path):
        super(WrappedUNISAL, self).__init__()
        self.input_wrapper = InputWrapper()
        self.unisal_model = unisal.model.UNISAL()
        self.unisal_model.load_state_dict(torch.load(sal_path))
        self.unisal_model.eval()  # Set to evaluation mode

    def forward(self, x):
        x = self.input_wrapper(x)
        return self.unisal_model(x)

In [4]:
# Prepare teh time series data and load saliency maps
timepoints = 15
salmapsps = 6
fps = 30
future_time = 200  # prediction in ms time
future_steps = int((fps * future_time) / 1000)
sal_frames_keep = int(timepoints / fps * salmapsps)
interval = int(fps / salmapsps)

sal_path = "../models/unisal.pth"
time_path = "../trained_models/conv_tsmixer_test_stadarized_6fituresteps_lr3/model_epoch_11_val_loss_0.1634.pth"  # conv_tsmixer_test_stadarized/model_epoch_9_val_loss_0.1763.pth"

unisal_model = WrappedUNISAL(sal_path)
# unisal_model.load_state_dict(torch.load(sal_path))
# unisal_model.eval()
unisal_model.to(device)

num_time_features = 4
conv_output_embeddings = 10
sequence_length = 15
prediction_length = future_steps
prediction_features = 2
time_model = Conv_tsmixer(
    num_time_features,
    conv_output_embeddings,
    sequence_length,
    prediction_length,
    prediction_features,
)
time_model.load_state_dict(torch.load(time_path))
time_model.eval()
time_model.to(device)


/home/cp/miniconda3/envs/access/lib/python3.9/site-packages/torch/functional.py:507: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3549.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Conv_tsmixer(
  (conv1): Conv2d(1, 16, kernel_size=(5, 5), stride=(1, 1))
  (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(16, 8, kernel_size=(5, 5), stride=(1, 1))
  (bn2): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=72, out_features=10, bias=True)
  (Dropout): Dropout(p=0.5, inplace=False)
  (tsmixer): TSMixer(
    (mixer_layers): Sequential(
      (0): MixerLayer(
        (time_mixing): TimeMixing(
          (norm): TimeBatchNorm2d(210, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (fc1): Linear(in_features=15, out_features=15, bias=True)
        )
        (feature_mixing): FeatureMixing(
          (norm_before): TimeBatchNorm2d(210, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      

In [5]:
# Example of preprocessing done outside the model
batch_size = 1
sequence_length = 15
image_sequence = torch.randn(batch_size, sequence_length, 288, 384, 3).cpu().numpy()

# Preprocess each image in the batch
preprocessed_images = []
for i in range(batch_size):
    for j in range(sequence_length):
        img = image_sequence[i, j]
        preprocessed_img = preprocess(img)
        preprocessed_images.append(preprocessed_img)

img_timeseries = torch.stack(preprocessed_images).to(device)
time_series_data = torch.randn(batch_size, sequence_length, 4).to(device)  # 4 is the number of original time features


In [6]:
img_timeseries[0].shape

torch.Size([3, 288, 384])

In [7]:
img_timeseries[0].unsqueeze(0)[0].shape

torch.Size([3, 288, 384])

In [8]:
img_timeseries.shape

torch.Size([15, 3, 288, 384])

In [9]:
with torch.no_grad():
    saliency_map = unisal_model(img_timeseries[0].unsqueeze(0))
    
    # Iterate over the 15 saliency maps and apply the process
    processed_maps = []

    for i in range(saliency_map.shape[1]):  # Iterate over the second dimension (15 saliency maps)
        smap = saliency_map[0, i, 0, :, :].cpu().numpy()  # Extract the 2D saliency map and convert to numpy
        smap_processed = sal_process(smap)    # Apply the processing function
        processed_maps.append(torch.tensor(smap_processed))  # Convert back to tensor
    # Stack back into a tensor with the same shape as the original
    processed_output = torch.stack(processed_maps).unsqueeze(0).to(device)

    pred = time_model(processed_output,time_series_data)

/home/cp/miniconda3/envs/access/lib/python3.9/site-packages/torch/nn/modules/conv.py:456: UserWarning: Applied workaround for CuDNN issue, install nvrtc.so (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:80.)
  return F.conv2d(input, weight, bias, self.stride,


In [10]:
processed_maps[0].min()

tensor(-1.)

In [11]:
pred.shape

torch.Size([1, 6, 2])

## Convert to mlcore


In [12]:
img_timeseries[0].unsqueeze(0).unsqueeze(0).shape

torch.Size([1, 1, 3, 288, 384])

In [14]:
unisal_model(img_timeseries[0].unsqueeze(0)).max()

tensor(-10.0998, device='cuda:0', grad_fn=<MaxBackward1>)

In [15]:
img_timeseries[0].unsqueeze(0).shape

torch.Size([1, 3, 288, 384])

In [16]:
import coremltools as ct

unisal_model_traced = torch.jit.trace(unisal_model,[img_timeseries[0].unsqueeze(0)]).to(device)

# Set the image scale and bias for input image preprocessing
scale = 1/(0.226*255.0)
bias = [- 0.485/(0.229) , - 0.456/(0.224), - 0.406/(0.225)]
# Using image_input in the inputs parameter:
# Convert to Core ML program using the Unified Conversion API.
sal_model = ct.convert(
    unisal_model_traced,
    convert_to="mlprogram",
    inputs=[ct.TensorType(name="fov_rgb",
                           shape=img_timeseries[0].unsqueeze(0).shape,
                           # scale=scale, bias=bias
                        )],
    outputs=[ct.TensorType(name="saliecy_maps")]
 )


Torch version 2.2.1+cu118 has not been tested with coremltools. You may run into unexpected errors. Torch 2.2.0 is the most recent version that has been tested.
Failed to load _MLModelProxy: No module named 'coremltools.libcoremlpython'
/tmp/ipykernel_8708/1735463241.py:6: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if x.shape != (1, 3, 288, 384):
/home/cp/Documents/Libra/access/eht_gaze_prediction/notebooks/../unisal/unisal/model.py:434: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  static = x.shape[1] == 1 or self.sources == ('SALICON',)
/home/cp/D

In [17]:
sal_model.user_defined_metadata["com.apple.coreml.model.preview.type"]="depthEstimation"

In [18]:
sal_model.save('/home/cp/Documents/Libra/access/eht_gaze_prediction/models/mlcore_unisal_tsmixer/unisal_1img_rgb_meta.mlpackage')

In [23]:
time_model_traced = torch.jit.trace(time_model,[processed_output,time_series_data]).to(device)

# Using image_input in the inputs parameter:
# Convert to Core ML program using the Unified Conversion API.
time_mlcore = ct.convert(
    time_model_traced,
    convert_to="mlprogram",
    inputs=[ct.TensorType(shape=processed_output.shape, name="processed_saliency_maps"),ct.TensorType(shape=time_series_data.shape, name="hmd_time_series")],
    outputs=[ct.TensorType(name="gaze_predictions")]
 )

Running MIL frontend_pytorch pipeline:   0%|          | 0/5 [00:00<?, ? passes/s]Saving value type of int64 into a builtin type of int32, might lose precision!
Saving value type of int64 into a builtin type of int32, might lose precision!
Saving value type of int64 into a builtin type of int32, might lose precision!
Saving value type of int64 into a builtin type of int32, might lose precision!
Saving value type of int64 into a builtin type of int32, might lose precision!
Saving value type of int64 into a builtin type of int32, might lose precision!
Saving value type of int64 into a builtin type of int32, might lose precision!
Saving value type of int64 into a builtin type of int32, might lose precision!
Saving value type of int64 into a builtin type of int32, might lose precision!
Saving value type of int64 into a builtin type of int32, might lose precision!
Saving value type of int64 into a builtin type of int32, might lose precision!
Saving value type of int64 into a builtin type of 

In [24]:
time_mlcore.save('/home/cp/Documents/Libra/access/eht_gaze_prediction/models/mlcore_unisal_tsmixer/tsmixer.mlpackage')